# Synthetic example: recover a known model

This notebook uses a deliberately simple case: we first define an `exp_shifted` model with known parameters, generate synthetic concentrations at a single date, and then ask PyAges to recover those parameters from noisy observations only.

The teaching value is direct: the `truth` is known. The outputs can therefore be read not only as numerical results, but also as a check of what the calibration is actually doing.

## Notebook roadmap

This notebook follows seven short steps:

1. review the synthetic-case parameters;
2. understand the calibration parameters;
3. generate the synthetic dataset;
4. understand the columns in the first table;
5. run the calibration;
6. read the summary figures and the recovery table;
7. inspect the expert view.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yaml
from IPython.display import Image, Markdown, display
from IPython.utils.capture import capture_output

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyages').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if not (ROOT / 'pyages').exists():
    raise RuntimeError('Run this notebook from the repository root or one of its subdirectories.')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXAMPLE_DIR = ROOT / 'examples' / 'synthetic' / 'lpm_recovery_single_date'
if str(EXAMPLE_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLE_DIR))

from synthetic_case import generate_synthetic_case
from run_lpm_recovery_single_date import main as run_example

expert_mode = True

## 1. Synthetic-case parameters

Before running anything, it is useful to review the settings that define the ground truth of the case: the model used to generate the concentrations, its parameters, the tracer list, the observation date, and the noise level added to the observations.

These settings can be edited directly in the example YAML files.

In [ ]:
generation_settings_path = EXAMPLE_DIR / 'generation' / 'generation_settings.yaml'
workflow_settings_path = EXAMPLE_DIR / 'lpm_recovery_single_date.yaml'

generation_settings = yaml.safe_load(generation_settings_path.read_text(encoding='utf-8'))
workflow_settings = yaml.safe_load(workflow_settings_path.read_text(encoding='utf-8'))

case_rows = [
    {'parameter': 'true model', 'value': generation_settings['lpm']['model_name']},
    {'parameter': 'true parameters', 'value': generation_settings['lpm']['parameters']},
    {'parameter': 'date', 'value': generation_settings['generation']['date']},
    {'parameter': 'tracers', 'value': ', '.join(generation_settings['generation']['tracers'])},
    {'parameter': 'relative noise level', 'value': generation_settings['generation']['relative_error']},
]

case_table = pd.DataFrame(case_rows)
display(case_table)

display(Markdown(f'Generation file: `{generation_settings_path}`'))
display(Markdown(f'Calibration file: `{workflow_settings_path}`'))

## 2. Calibration parameters

Before execution, it is also useful to review the settings that control the calibration itself. The table below lists the main parameters for the `prior` stage and the `posterior` stage, together with their practical role.

In [ ]:
calibration_specs = [
    (
        'prior',
        'run.reachable_concentrations',
        workflow_settings['run']['reachable_concentrations'],
        'enables the pre-calibration reachable-space computation used in the observations-vs-prior figure',
    ),
    (
        'prior',
        'reachable_concentrations.nmodels',
        workflow_settings['reachable_concentrations']['nmodels'],
        'number of sampled models used to approximate reachable space; larger means a smoother figure but a longer run',
    ),
    (
        'prior',
        'run.objective_function',
        workflow_settings['run']['objective_function'],
        'enables the objective-function grid used in the final summary figure',
    ),
    (
        'prior',
        'objective_function.nmodels',
        workflow_settings['objective_function']['nmodels'],
        'size of the sampled objective grid; larger means a finer background map',
    ),
    (
        'posterior',
        'calibration_metropolis_hastings.nstep',
        workflow_settings['calibration_metropolis_hastings']['nstep'],
        'number of MH steps; larger means a denser posterior cloud but a longer run',
    ),
    (
        'posterior',
        'calibration_metropolis_hastings.prior_option',
        workflow_settings['calibration_metropolis_hastings']['prior_option'],
        'controls whether the prior contribution is explicitly included in the MH score',
    ),
    (
        'posterior',
        'calibration_metropolis_hastings.likelihood',
        workflow_settings['calibration_metropolis_hastings']['likelihood'],
        'uses the likelihood of the noisy observations; it should stay enabled for this case',
    ),
    (
        'posterior',
        'calibration_metropolis_hastings.monitor',
        workflow_settings['calibration_metropolis_hastings']['monitor'],
        'prints a textual progress monitor during the chain; mainly useful for debugging',
    ),
    (
        'posterior',
        'calibration_metropolis_hastings.display_traj',
        workflow_settings['calibration_metropolis_hastings']['display_traj'],
        'displays the chain trajectory during execution; mainly useful in expert mode',
    ),
]

calibration_table = pd.DataFrame(
    calibration_specs,
    columns=['phase', 'parameter', 'value', 'role'],
)
display(calibration_table)

The first calibration settings to adjust are usually `calibration_metropolis_hastings.nstep` and the sampling sizes `reachable_concentrations.nmodels` / `objective_function.nmodels`.

By contrast, the `relative noise level` is not a calibration parameter: it belongs to dataset generation and controls how difficult the inverse problem is.

## 3. Generate the synthetic case

Generation produces two different things:

- the true concentrations, computed directly from the synthetic model;
- the observed concentrations, which are the true concentrations perturbed by controlled noise.

The first table below compares these two information levels directly.

In [ ]:
case = generate_synthetic_case()
truth = case.truth_payload

truth_table = pd.DataFrame(
    [
        {'parameter': name, 'true_value': value}
        for name, value in truth['lpm']['parameters'].items()
    ]
)

comparison = case.true_frame[['element', 'concentration']].rename(
    columns={'concentration': 'true_concentration'}
).merge(
    case.observed_frame[['element', 'concentration', 'error']].rename(
        columns={'concentration': 'observed_concentration', 'error': 'uncertainty'}
    ),
    on='element',
    how='left',
)
comparison['absolute_difference'] = comparison['observed_concentration'] - comparison['true_concentration']
comparison['relative_difference_%'] = 100.0 * comparison['absolute_difference'] / comparison['true_concentration']

display(Markdown('**True parameters used to generate the data**'))
display(truth_table)

display(Markdown('**True concentrations and noisy observations**'))
display(comparison.round(3))

### How to read the table columns

- `element`: the tracer being considered (`CFC11`, `CFC12`, `CFC113`, `SF6`).
- `true_concentration`: the concentration computed directly from the true synthetic model.
- `observed_concentration`: the concentration actually passed to calibration after adding noise.
- `uncertainty`: the assumed measurement error in the synthetic case; here it is proportional to the true concentration.
- `absolute_difference`: the raw difference between the noisy observation and the true concentration.
- `relative_difference_%`: the same difference, normalized by the true concentration and expressed as a percentage.

This table makes it immediately clear what the calibration must explain: not the ideal truth, but a noisy observation.

## 4. Run the calibration

The cell below:

- regenerates the same synthetic observations;
- runs the single-date workflow;
- writes the figures and tables to the results directory.

The figures are not displayed here to avoid duplicates: they are reloaded cleanly in the next section.

In [ ]:
with capture_output():
    results_dir = run_example(force_inline=True)

display(Markdown('Calibration completed. The figures are shown in the next section.'))
display(Markdown(f'**Results directory:** `{results_dir}`'))
results_dir

## 5. Read the calibration results

The three figures below summarize the essentials:

- how the observations sit relative to the reachable space and the posterior cloud;
- how the parameters are recovered;
- how the objective function connects the `prior grid`, `posterior samples`, and the true parameters.

In [ ]:
display(Image(filename=str(results_dir / "01_data_model_space.png"), width=980))
display(Markdown("""**How to read this figure**

- the black points are the observations given to calibration;
- the reference diamond marks the true synthetic model;
- the light background shows the explored reachable space before calibration (`prior reachable space`);
- the transparent blue cloud shows the posterior samples.

With four tracers, the figure is arranged as a 2 x 2 grid to remain readable."""))

In [ ]:
display(Image(filename=str(results_dir / "02_parameter_summary.png"), width=900))
display(Markdown("""**How to read this figure**

- each histogram shows the posterior distribution of one parameter;
- the vertical reference line marks the true value used to generate the data.

The result is good when the true value falls inside the dense part of the posterior distribution, even if it is not exactly at the maximum."""))

In [ ]:
display(Image(filename=str(results_dir / "03_objective_summary.png"), width=980))
display(Markdown("""**How to read this figure**

- the colored background shows the objective function computed on a `prior grid`;
- the transparent blue points show the Metropolis-Hastings posterior samples;
- the black marker shows the true synthetic parameters.

The goal here is to check that the favorable part of the objective function covers the neighborhood of the true parameters and that the posterior concentrates there."""))

## 6. Compare true and estimated parameters

The table below summarizes parameter recovery. It compares the true value, the posterior mean, the posterior uncertainty, and the residual difference.

In [ ]:
recovery = pd.read_table(results_dir / 'parameter_recovery_summary.txt', sep='	')
recovery = recovery.rename(
    columns={
        'estimated_mean': 'posterior_mean',
        'estimated_std': 'posterior_std',
        'difference': 'estimated_minus_true',
    }
)
recovery['relative_difference_%'] = 100.0 * recovery['estimated_minus_true'] / recovery['true_value']
recovery['true_within_1_sigma'] = recovery['estimated_minus_true'].abs() <= recovery['posterior_std']
display(recovery.round(3))

Exact equality between the true value and the posterior mean is not the target. With noisy data, the right question is instead: does the true value remain close to the center of the posterior, and is the observed difference of the same order as the estimated uncertainty?

## 7. Expert mode

Because `expert_mode = True` by default, the notebook shows a more analytical view here: posterior solutions are projected directly onto the objective function shown as an interpolated colored surface.

This figure helps assess whether the `Metropolis_Hastings` chain concentrates in the low-objective region, and how that region sits relative to the true parameters.

In [ ]:
if expert_mode:
    from pyages.workflows.plots import plot_objective_solution_map

    posterior = pd.read_table(
        results_dir / "Metropolis_Hastings" / "lpm_dist_calibrated.txt",
        sep="	",
        index_col=0,
    )
    objective_grid = pd.read_table(results_dir / "objective_function_grid.txt", sep="	")
    reference_params = {
        name: float(value)
        for name, value in truth["lpm"]["parameters"].items()
    }
    plot_objective_solution_map(
        objective_frame=objective_grid,
        posterior_frame=posterior,
        param_names=list(reference_params.keys()),
        reference_params=reference_params,
        reference_label="True parameters",
        title="Expert view: posterior solutions colored by objective value",
    )
    display(Markdown("""**How to read this figure**

- the interpolated colored background represents the objective function sampled on the `prior` grid;
- the colored points are the `posterior` solutions, shaded by their own objective value;
- the white star marks the best posterior solution;
- the black diamond marks the true synthetic parameters.

Ideally, the posterior cloud should tighten inside the lowest-objective region."""))